# Advisor results walkthrough
INTERNAL ONLY: complete multi-variant comparisons and separate CFO model; not the advisor presentation.
Run from the repository (or notebooks directory) after bundle provisioning. This notebook loads saved results, performs small identity checks and displays previously regenerated figures. No training or BER evaluation. See README for commands.


In [ ]:
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from advisor.core import configure
from advisor.artifacts import load_reference
from advisor.delay_doppler import bridge
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'configs/model.json').is_file())
artifacts = Path(os.environ.get('ADVISOR_ARTIFACTS', str(root/'local/artifacts')))
if not artifacts.exists():
    raise FileNotFoundError('Provision local bundle first; see README. No old source-project paths are used.')
configure()


## H → W → A
Columns of W synthesize symbols. Original ordering is authoritative; saved group permutation is an additional fixed view.


In [ ]:
data = np.load(artifacts/'focused/data/channels.npz', allow_pickle=False)
w = load_reference(artifacts)
h = torch.from_numpy(data['H'])
a = w.mH @ h @ w
b = bridge(w,h)
print('H,W,A:', tuple(h.shape),tuple(w.shape),tuple(a.shape),a.dtype,a.device)
print('unitarity:',float(torch.linalg.matrix_norm(w.mH@w-torch.eye(64))))
print('complex spectral reconstruction:',float(torch.linalg.matrix_norm(a-b['A_spectral'])/torch.linalg.matrix_norm(a)))
print('paths:',data['tau'],data['nu'], 'normalization:',data['normalization'])


## Core figures
Default25dB view only; full numerical grids retained through30/40dB. Main raw slicer; fixed-Q and full receivers separated. Generate figures with the README command first.


In [ ]:
from IPython.display import display, Image
for name in ['channel_matrices_native','basis_time','basis_frequency','ber_QPSK_raw','ber_16QAM_raw']:
    path = root/'local/figures'/f'{name}.png'
    if path.exists(): display(Image(filename=str(path), width=1000))
    else: print('Generate first:',path.name)
summary = pd.read_csv(artifacts/'focused/ber_summary.csv')
display(summary[(summary.snr_db==25)&(summary.slicer=='raw')][['modulation','waveform','receiver','errors','bits','ber','ci_low','ci_high']])


## Separate residual CFO
Genuine rectangular W64x45, analysis R45x64. Positive-sign CFO; historical float32/complex64. Operator/acceptance qualification only; saved Eb/N0 BER not rerun.


In [ ]:
from advisor.residual_cfo import load_qualified, effective, operator
wc,rc,checks = load_qualified(artifacts)
ac = effective(wc,rc,.1)
print('W,R,H,A:',wc.shape,rc.shape,operator(.1).shape,ac.shape)
print('clean identity:',checks['clean_identity_loss'])
print('file SHA:',checks['checkpoint_file_sha256'])


## Supplementary evidence and limits
Completed N64 training, intervention and structural-transfer evidence is indexed in docs/STUDIES.md. Mean-LMMSE MSE invariance does not imply BER invariance. Frequency groups are approximate/leaky, not an exact analytical construction. Original DD .pt unavailable; verified export is available. Active N128/N256 results excluded.


In [ ]:
display(pd.read_csv(artifacts/'supplementary/main/training_selected_summary.csv'))
display(pd.read_csv(artifacts/'supplementary/main/impairment_transfer.csv').query("strength == 1 and waveform == 'T0_tail'"))
